# Day 15 · A search engine in 30 lines of NumPy

**First-principles question:** What does a vector database actually do when you query it?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import time
import numpy as np
from lab.acme import load_kb
from lab.llm import embed

kb = load_kb()
chunks = [f"{a['title']}. {a['text']}" for a in kb]     # one chunk per article
print(len(chunks), "chunks")

## 1 · Embed every chunk once

In [ ]:
# TODO: M = a numpy array of embed(chunks), then make every row length 1
#       (divide by np.linalg.norm(M, axis=1, keepdims=True))
M = None
print(M.shape)

## 2 · Search with one matrix multiply

In [ ]:
def search(query, k=3):
    """The k best chunks for the query: [(article id, title, score), ...], best first."""
    # TODO: q = embed([query])[0] as a numpy array, made length 1
    #       scores = M @ q
    #       top = the indexes of the k highest scores (np.argsort(-scores)[:k])
    #       return [(kb[i]["id"], kb[i]["title"], float(scores[i])) for i in top]
    ...

search("why is there an extra 100 rupees on my bill")

## 3 · The same, with a loop
Proof that `M @ q` is just many dot products.

In [ ]:
def search_loop(query, k=3):
    q = np.array(embed([query])[0]); q = q / np.linalg.norm(q)
    scores = []
    for i, row in enumerate(M):
        # TODO: append (score, i) where score = the dot product of row and q (a loop or sum(...))
        pass
    scores.sort(reverse=True)
    return [(kb[i]["id"], kb[i]["title"], float(s)) for s, i in scores[:k]]

QUERIES = ["why is there an extra 100 rupees on my bill", "net kab chalega", "I want to stop my connection",
           "moving to Wakad next week", "someone asked me for an OTP"]
for query in QUERIES:
    fast, slow = search(query), search_loop(query)
    same = np.allclose([s for _, _, s in fast], [s for _, _, s in slow])     # same scores, rank by rank
    print(f"{'✅' if same else '❌'} {query!r}: {[(i, round(s, 3)) for i, _, s in fast]}")

## 4 · Is matrix multiply really faster?

In [ ]:
big = np.random.default_rng(0).normal(size=(100_000, M.shape[1]))
q = big[0]
t0 = time.perf_counter(); big @ q; t_matrix = time.perf_counter() - t0
t0 = time.perf_counter(); [float(np.dot(row, q)) for row in big[:10_000]]; t_loop = (time.perf_counter() - t0) * 10
print(f"100,000 vectors: matrix {t_matrix * 1000:.1f} ms · loop about {t_loop * 1000:.0f} ms")

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert M.shape[0] == len(chunks) and np.allclose(np.linalg.norm(M, axis=1), 1), "Step 1: unit rows"
r = search("forgot my wifi password", k=4)
assert len(r) == 4 and all(r[i][2] >= r[i + 1][2] for i in range(3)), "search() returns k results, best first"
for query in QUERIES:
    fast, slow = [x[2] for x in search(query)], [x[2] for x in search_loop(query)]
    assert np.allclose(fast, slow), f"Loop and matrix disagree on {query!r}"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")